In [ ]:
import numpy as np
import pandas as pd
from openai import OpenAI
from scipy.spatial import distance
from ast import literal_eval

In [ ]:
client = OpenAI(api_key=" ")

In [ ]:
def get_embedding(text, model="text-embedding-ada-002"):
    edited_text = text.replace("\n", " ")   
    return client.embeddings.create(input=[edited_text], model=model).data[0].embedding

In [ ]:
def split(text):
    start = text.find('(')
    if start == -1:
        return text, ""
    end = text.find(')', start + 1)
    if end == -1:
      return text, ""
    return text[:start], text[start+1:end]

For each patient, the following code was used to determine the top 3 active problems based on embedding distance and compare them to the top 3 human determined active problems

In [ ]:
#read csv data
datapath = " "
df_ap = pd.read_csv(datapath + " ") #df for human annotated active problems
df_ros = pd.read_csv(" ") #df for ros 
df_visits = pd.read_csv(" ") #df for patient visit encounters
distances = pd.read_csv(" ") #df for patient visit encounter embedding distances

In [ ]:
df_v2 = pd.DataFrame(columns=['Visit', 'AI Top ROS', 'Human Top ROS', 'AI Second ROS', 'Human Second ROS', 'AI Third ROS', 'Human Third ROS'])

#separate active problems from (ros) for first set
#iterate through column
for i in range(len(df_ap["Active Problem 1"])):
    cell = df_ap.at[i, "Active Problem 1"]
    if isinstance(cell, str) and cell.strip() != "":                                #if true, there is text in cell
        ap = df_ap.at[i, "Active Problem 1"]                                       
        annotation, ros = split(ap)                                                 #split text                           
        df_v2.at[i, "Human Top ROS"] = ros
#for second set
for i in range(len(df_ap["Active Problem 2"])):
    cell = df_ap.at[i, "Active Problem 2"]
    if isinstance(cell, str) and cell.strip() != "":
        ap = df_ap.at[i, "Active Problem 2"]
        annotation, ros = split(ap)
        df_v2.at[i, "Human Second ROS"] = ros
#for third
for i in range(len(df_ap["Active Problem 3"])):
    cell = df_ap.at[i, "Active Problem 3"]
    if isinstance(cell, str) and cell.strip() != "":
        ap = df_ap.at[i, "Active Problem 3"]
        annotation, ros = split(ap)
        df_v2.at[i, "Human Third ROS"] = ros

In [ ]:
#populate table 
#pick AI top three ROS based on distance
numofvisits = len(df_visits)
rosnum = len(df_ros)
for i in range((numofvisits)):
    smallest_vals = distances.iloc[:,i].nsmallest(3)
    min_indices = smallest_vals.index
    df_v2.at[i, "Visit"] = distances.columns[i]
    df_v2.at[i, "AI Top ROS"] = df_ros.at[min_indices[0],"Organ System"]
    df_v2.at[i, "AI Second ROS"] = df_ros.at[min_indices[1], "Organ System"]
    df_v2.at[i, "AI Third ROS"] = df_ros.at[min_indices[2], "Organ System"]

In [ ]:
#save files 
df_v2.to_csv(datapath + " ", index = False)
df_v2.to_excel(datapath + " ", index = False)